# Line Fitting — Watch a Model Learn From Data
**Demystifying AI — Industry Event, May 2026**

Welcome — your first hands-on moment of the session.

## First: what is this thing?

If you've never opened a notebook before, two minutes of orientation:

**What you're looking at is a *Jupyter notebook*, opened in *Google Colab*.** A notebook is a document that mixes blocks of writing (like this one) with blocks of code. The code blocks have a little ▶ button — clicking it runs that code on Google's computer and shows the result right below. Think of it as a lab notebook for software: code, results, and notes interleaved on one page.

**Google Colab** is Google's free service for running notebooks in the browser. You don't install anything. You don't need to know Python. Google hands every attendee a temporary virtual computer to run the demo on. Close the tab when you're done; nothing to clean up.

**What you're about to do:** click `Runtime → Run all` once. Two animations will appear — one showing a line gradually settling onto its data as more data arrives, then another showing the same line getting yanked around by a handful of bad data points. **Each animation has a play button — click it when Mark says go.**

**You do not need to read or understand the code.** Every block is commented for the curious, but the demo is in the animations.

> **No GPU needed for this one.** Default Colab runtime (CPU) is fine — save your GPU allocation for the nanoGPT demo later in the session.

## What's in this notebook

- **Setup** — imports and a few knobs
- **Helpers** — three small functions: build fake data, fit a line, draw a plot
- **Phase 1 animation** — watch a line stabilize as data grows from 2 points to 1,000
- **Phase 2 animation** — inject 3 anomalies into a clean dataset; compare two fitting algorithms
- **Recap** — what this means and where the talk goes next


In [ ]:
# -------------------------------------------------------------------
# Setup — imports and the knobs that control the demo
# -------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, HuberRegressor


# === The "ground truth" line ===
# We KNOW the real relationship is y = 2.5x + 4. We're going to generate
# noisy data around that line, then see how well different fitting
# algorithms recover the true slope (m) and intercept (b).
# This is the simplest possible version of "training a model on data."
TRUE_M = 2.5      # true slope
TRUE_B = 4.0      # true y-intercept
NOISE_STD = 3.0   # how noisy each data point is around the true line

# Seed the random number generator so everyone in the room sees the
# SAME data on their screen (otherwise each laptop would draw different
# points and the narration wouldn't match).
RNG = np.random.default_rng(42)

print("Setup loaded. True line we're hunting: y = 2.5x + 4.")
print("Now run the next cell, knucklehead.")


In [ ]:
# -------------------------------------------------------------------
# Three small helper functions — make data, fit a line, draw a plot
# -------------------------------------------------------------------


def make_data(n, noise_std=NOISE_STD):
    """Generate n points scattered around the true line y = TRUE_M * x + TRUE_B."""
    x = np.linspace(0, 20, n)              # n evenly-spaced x values from 0 to 20
    noise = RNG.normal(0, noise_std, n)    # Gaussian noise added to each y
    y = TRUE_M * x + TRUE_B + noise
    return x, y


def fit_ols(x, y):
    """Fit a line using ORDINARY LEAST SQUARES — the textbook regression.

    OLS = "find the m and b that minimize the sum of SQUARED errors."
    The squaring is the key detail: one big error counts more than many
    small ones. That's why OLS is sensitive to outliers (Phase 2 shows this).
    """
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def fit_huber(x, y):
    """Fit a line using HUBER REGRESSION — a "robust" alternative to OLS.

    Huber treats small errors like OLS does (squared), but treats LARGE
    errors LINEARLY instead of squaring them. That one change makes it
    much harder for a handful of bad points to drag the line off course.
    Used in any production setting where you expect dirty data.
    """
    model = HuberRegressor().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def draw(ax, x, y, m, b, title, extra_line=None, y_limits=None):
    """Draw one panel: scatter the points, plot the fit, plot the truth."""
    ax.clear()
    ax.scatter(x, y, s=30, alpha=0.7, label=f"data (n={len(x)})")

    # The line our model fit (red, solid)
    xs = np.array([0, 20])
    ax.plot(xs, m * xs + b, "r-", lw=2.2,
            label=f"OLS fit: y = {m:.2f}x + {b:.2f}")

    # Optional second line (used in Phase 2 to show Huber alongside OLS)
    if extra_line is not None:
        m2, b2, label2 = extra_line
        ax.plot(xs, m2 * xs + b2, "g--", lw=2.2, label=label2)

    # The true line we know we should be recovering (black dotted)
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1, alpha=0.6,
            label=f"truth: y = {TRUE_M}x + {TRUE_B}")

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 20)
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)


print("Helpers ready: make_data, fit_ols, fit_huber, draw.")
print("Phase 1 is next — that's the dramatic one. Go run it.")


## Phase 1 — Watch gradient descent search for the answer

We give the model 50 noisy data points and a wildly-wrong starting line (flat at zero — slope 0, intercept 0). Then we run **gradient descent**, the same algorithm that trains every neural network on Earth, including GPT-4. Each frame is one optimization step.

**Two panels in sync:**

- **Left** — the data (blue dots) and the line as the algorithm currently sees it (red, moving). The dotted black line is the true answer we're trying to find.
- **Right** — a *map* of how wrong every possible `(slope, intercept)` would be. Bright yellow = very wrong. Dark purple = nearly perfect. The **white dot** is where the algorithm is right now. The **red star** is the global minimum — the answer. Watch the dot roll downhill toward it. That's where "gradient *descent*" gets its name.

**What to watch for:**

- The red line **flails** in the first 10 steps, then settles
- The white dot on the right panel traces a **path** as it crawls toward the star
- The loss number in the title falls from ~500 to ~5
- Most of the drama is in the first 30 frames; the rest is fine-tuning

> **Hit ▶ (play) on the animation widget below when Mark says go.** You can pause, step frame-by-frame, or drag the scrubber to dwell on any moment.


In [ ]:
# -------------------------------------------------------------------
# Phase 1 — Gradient descent in motion (two-panel animation)
#
# Left panel:  data + the line being trained
# Right panel: 2D loss landscape (m on x-axis, b on y-axis,
#              color = loss). A dot rolls downhill toward the minimum.
#
# This is what "training" actually IS. The algorithm doesn't know the
# answer; it just measures how wrong it is and steps slightly toward
# less-wrong. Same procedure that trains GPT-4, just on 2 parameters
# instead of 1.76 trillion.
# -------------------------------------------------------------------

from matplotlib import rc
from matplotlib.animation import FuncAnimation
from matplotlib.colors import LogNorm

# Render animations as inline HTML5 players (Colab's standard pattern)
rc("animation", html="jshtml")
rc("animation", embed_limit=50)  # MB cap on embedded animation size

# === Fixed 50-point dataset (same seed = same data on every laptop) ===
RNG = np.random.default_rng(42)   # reset RNG so this phase is self-contained
x_fixed, y_fixed = make_data(50)
N = len(x_fixed)

# === Pre-compute the loss landscape ===
# For every (m, b) in a grid, compute the loss on our fixed dataset.
# Vectorized: shape gymnastics let us evaluate the whole grid at once.
m_grid = np.linspace(-1, 5, 100)
b_grid = np.linspace(-15, 20, 100)
M, B = np.meshgrid(m_grid, b_grid)             # both shape (100, 100)
# Broadcast to (100, 100, 50) and reduce over the 50 data points
predictions = M[:, :, None] * x_fixed[None, None, :] + B[:, :, None]
errors = predictions - y_fixed[None, None, :]
loss_surface = (errors ** 2).sum(axis=2) / (2 * N)

# === Pre-compute the gradient-descent trajectory ===
# We record (m, b, loss) at every step so the animation just replays
# pre-computed values — no live computation during playback.
#
# Note on the two learning rates: the gradient for m involves
# multiplication by x (mean ~10 here), so it's intrinsically ~10x
# larger than the gradient for b. With a single learning rate, m
# converges fast while b lags. A larger lr_b balances that — same
# trick Adam/RMSprop do automatically at scale.
trajectory = []
m, b = 0.0, 0.0    # initial guess: flat line at y = 0
lr_m = 0.002       # learning rate for slope
lr_b = 0.04        # learning rate for intercept (20x larger — see note above)
n_steps = 100

for step in range(n_steps + 1):
    y_pred = m * x_fixed + b
    residuals = y_pred - y_fixed
    loss = (residuals ** 2).sum() / (2 * N)
    trajectory.append((m, b, loss))

    # The gradient of mean-squared-error loss with respect to m and b.
    # Pure calculus, the exact same math behind every neural net optimizer.
    grad_m = (residuals * x_fixed).sum() / N
    grad_b = residuals.sum() / N

    # Step downhill (opposite the gradient), with per-parameter learning rates
    m -= lr_m * grad_m
    b -= lr_b * grad_b

m_path = [t[0] for t in trajectory]
b_path = [t[1] for t in trajectory]

# === Build the two-panel animation ===
fig, (ax_data, ax_loss) = plt.subplots(1, 2, figsize=(16, 7))


def update_phase1(frame):
    m_now, b_now, loss_now = trajectory[frame]

    # ----- LEFT PANEL: data + current fit line -----
    draw(ax_data, x_fixed, y_fixed, m_now, b_now,
         f"step {frame}  |  m={m_now:.2f}  b={b_now:.2f}  loss={loss_now:.2f}",
         y_limits=(-10, 70))

    # ----- RIGHT PANEL: loss landscape + trajectory -----
    ax_loss.clear()
    # Contour-fill the loss surface (log color scale so the bottom of
    # the valley is visible instead of being swamped by the peaks)
    ax_loss.contourf(M, B, loss_surface, levels=30, cmap="viridis",
                     norm=LogNorm(vmin=loss_surface.min() + 0.01,
                                  vmax=loss_surface.max()))
    # Path travelled so far (white line)
    ax_loss.plot(m_path[:frame + 1], b_path[:frame + 1],
                 "w-", lw=2.5, alpha=0.9)
    # Current position (white-edged dot)
    ax_loss.plot(m_now, b_now, "wo", markersize=14,
                 markeredgecolor="black", markeredgewidth=1.5)
    # True minimum (red star — the answer we're hunting)
    ax_loss.plot(TRUE_M, TRUE_B, "r*", markersize=24,
                 markeredgecolor="white", markeredgewidth=1.5,
                 label=f"target: m={TRUE_M}, b={TRUE_B}")
    ax_loss.set_xlabel("m (slope)")
    ax_loss.set_ylabel("b (intercept)")
    ax_loss.set_title(f"loss landscape  |  white dot = current guess, star = answer",
                      fontsize=11)
    ax_loss.set_xlim(-1, 5)
    ax_loss.set_ylim(-15, 20)
    ax_loss.legend(loc="upper right", fontsize=9)
    ax_loss.grid(alpha=0.2, color="white", linewidth=0.5)


anim_phase1 = FuncAnimation(
    fig, update_phase1, frames=len(trajectory), interval=120, repeat=False
)

plt.close(fig)
anim_phase1


**What you just watched:**

- The model started with a flat line — knowing **nothing**. The dot on the loss landscape started at the high-loss yellow region.
- Each frame, the algorithm did three things: (1) measured how wrong it was, (2) figured out which direction to nudge m and b to reduce error, (3) took a small step that direction.
- The line tilted and slid until it landed on the data. The dot rolled downhill until it landed on the star.
- That's **gradient descent.** Same algorithm that trains GPT-4. Same procedure. The only differences at frontier scale are: more parameters (1.76 trillion instead of 2), more data (the internet instead of 50 points), and more compute (a $100M data center instead of your laptop's browser tab).

> *"That's training. Not magic, not lookup, not memorization. A computer measuring how wrong it is and taking a step in a less-wrong direction, millions of times in a row. You just watched it happen in 100 steps on 2 parameters. Frontier models do the same thing on a trillion parameters for months. Same procedure."*


## Phase 2 — Bad data tilts the line

Real data is messy. Sensor glitches, typos, fat-fingered entries. What happens when a clean 100-point dataset picks up a few wildly-wrong points?

Below, we start with the clean dataset and progressively inject 1, 2, then 3 anomalies. We fit the same data two different ways:

- **OLS** (red, solid) — what most regressions use by default. Sensitive to outliers.
- **Huber** (green, dashed) — a "robust" alternative that down-weights outliers instead of squaring them.

**Hit ▶ on the animation below.** Watch the red line tilt. Watch the green line stay put.


In [ ]:
# -------------------------------------------------------------------
# Phase 2 — Animated anomaly injection, OLS vs Huber
# Each frame adds one more anomaly to a clean 100-point dataset.
# Output is an inline HTML5 player with play/pause/scrub controls.
# -------------------------------------------------------------------

# Start with a clean 100-point dataset
x_clean, y_clean = make_data(100)

# Three anomalies — chosen to be obviously, comically wrong
# (a real-world sensor glitch or data-entry typo would look similar)
anomalies_x = np.array([3.0, 8.0, 14.0])
anomalies_y = np.array([95.0, -35.0, 120.0])

fig, ax = plt.subplots(figsize=(11, 7))


def update_phase2(k):
    # k = number of anomalies injected so far (0, 1, 2, 3)
    xk = np.concatenate([x_clean, anomalies_x[:k]])
    yk = np.concatenate([y_clean, anomalies_y[:k]])

    # Fit OLS to the (clean + k anomalies) data
    m_ols, b_ols = fit_ols(xk, yk)

    # Also fit Huber once there's at least one anomaly to ignore
    extra = None
    if k >= 1:
        m_h, b_h = fit_huber(xk, yk)
        extra = (m_h, b_h, f"Huber (robust): y = {m_h:.2f}x + {b_h:.2f}")

    label = "anomaly" if k == 1 else "anomalies"
    draw(ax, xk, yk, m_ols, b_ols,
         f"Phase 2 — {k} {label} added (N = {len(xk)})",
         extra_line=extra, y_limits=(-50, 140))


# Slower interval here (2500 ms) so the room has time to register
# the tilt before the next anomaly lands.
anim_phase2 = FuncAnimation(
    fig, update_phase2, frames=len(anomalies_x) + 1, interval=2500, repeat=False
)

plt.close(fig)
anim_phase2


**What you should be seeing above:**

- **0 anomalies:** OLS sits right on top of the truth. No drama.
- **1 anomaly:** ONE bad point pulled the red OLS line off course. Huber barely moved.
- **2 anomalies:** OLS now visibly tilted. Huber still on the truth.
- **3 anomalies:** OLS is meaningfully wrong. Huber is still right.

> *"The algorithm determines how your model fails when reality is messy. When a vendor shows you a benchmark, they've shown you the red line on clean data. The real question — the one that predicts whether their model survives your production environment — is what happens when you hand it your messy data."*

---

## Where this goes next

Everything you just watched was **two numbers** — m and b — being adjusted to fit data. GPT-4 has **1.76 trillion** of those numbers. The adjustment procedure is the same. The only things that change are the function's shape, the parameter count, and how much compute you throw at it.

Keep that image in your head for the rest of the talk — because every architecture we're about to discuss (tokenization, embeddings, attention, transformers, the nanoGPT demo you'll run later) is a more elaborate version of this exact picture.
